# Три замера горячего поиска Monoforest

Индекс копируется в RAM и открывается один раз. Для одного и того же QUERY
заранее готовятся признаки и запрос, затем выполняется один явный прогрев
и три измеряемых поиска. Главная метрика — lucene_search_ms.
Подготовка, прогрев, explain и получение документов показаны отдельно.
Это повтор одного подготовленного запроса, а не скорость обработки нового текста.
Один прогрев не гарантирует завершения всей JIT-компиляции; сравнивайте все три замера.

После git pull пересоберите JAR: `mvn -Prunner -DskipTests clean package`.
Если notebook запущен из папки notebooks в репозитории, задайте
`JAR = Path("../target/monoforest-0.1.0-SNAPSHOT-runner.jar")`.
Проверьте пути к JAR, модели и индексу в следующей ячейке.


In [ ]:
from pathlib import Path
import json
import subprocess
import tempfile

JAVA = "/usr/lib/jvm/java-1.11.0-openjdk-amd64/bin/java"
HEAP = "4g"
JAR = Path("monoforest-0.1.0-SNAPSHOT-runner.jar")
INDEX = Path("/home/anglukhikhan/unpacked/lucene_index_positions")
MODEL = Path("linear_monomials.json")
QUERY = "grammar terms medicine"
TOP_K = 10
THRESHOLD = 3.0
REPEAT = 3
HOT = True  # одна подготовка + один прогрев + REPEAT измеряемых запусков
RAM_ROOT = "/run/jupyter-monitor/monoforest-anglukhikhan"  # None — исходный индекс


In [ ]:
if RAM_ROOT is not None:
    Path(RAM_ROOT).mkdir(mode=0o700, exist_ok=True)

command = [
    JAVA, f"-Xmx{HEAP}", "-jar", str(JAR),
    "--index", str(INDEX), "--model", str(MODEL), "--query", QUERY,
    "--top-k", str(TOP_K), "--threshold", str(THRESHOLD),
    "--repeat", str(REPEAT), "--hot", str(HOT).lower(),
]
if RAM_ROOT is not None:
    command += ["--ram-root", RAM_ROOT]

print("Загрузка индекса, подготовка и запуск измерений…", flush=True)
# JSON идёт во временный файл, чтобы большие тексты документов не блокировали pipe.
# Сообщения stderr показываются сразу по мере выполнения.
with tempfile.TemporaryFile(mode="w+", encoding="utf-8") as output:
    errors = []
    with subprocess.Popen(command, stdout=output, stderr=subprocess.PIPE,
                          text=True, encoding="utf-8") as process:
        for line in process.stderr:
            print(line, end="", flush=True)
            errors.append(line)
        returncode = process.wait()
    output.seek(0)
    raw_response = output.read()
if returncode:
    raise RuntimeError("".join(errors) or raw_response)
response = json.loads(raw_response)
if "runs" not in response:
    raise RuntimeError("Пересоберите JAR: в этой версии ещё нет отдельных замеров.")


In [ ]:
print(f"Режим: {response['mode']}")
if 'index_bytes' in response:
    print(f"Размер RAM-копии: {response['index_bytes'] / 1024**3:.2f} GiB")
print(f"Копирование в RAM: {response['copy_to_ram_ms'] / 1000:.3f} с")
print(f"Открытие модели и индекса: {response['open_model_and_index_ms'] / 1000:.3f} с")
if 'prepare_query_ms' in response:
    print(f"Подготовка запроса и фичей (один раз): {response['prepare_query_ms'] / 1000:.3f} с")
if 'warmup' in response:
    w = response['warmup']
    print(f"Прогрев: {w['search_ms'] / 1000:.3f} с всего; "
          f"из них поиск {w['timings_ms']['lucene_search_ms'] / 1000:.3f} с")

print("\nВсе значения в секундах. Поиск = только выполнение Lucene search.")
print(f"{'Запуск':>6} {'Подготовка':>11} {'Поиск':>11} {'Explain':>11} "
      f"{'ID':>11} {'Документы':>11} {'Прочее':>11} {'Всего':>11}")
for run in response['runs']:
    t = run['timings_ms']
    values = [t[k] / 1000 for k in ['build_query_ms', 'lucene_search_ms',
              'explain_ms', 'fetch_ids_ms', 'fetch_documents_ms', 'other_ms']]
    print(f"{run['run']:>6}" + ''.join(f"{v:>12.3f}" for v in values)
          + f"{run['search_ms'] / 1000:>12.3f}")

if response['benchmark_mode'] == 'hot_prepared_query':
    print("\nГорячий поиск, три отдельных замера (без подготовки, explain и документов):")
    for run in response['runs']:
        ms = run['timings_ms']['lucene_search_ms']
        print(f"Запуск {run['run']}: {ms:.3f} мс = {ms / 1000:.3f} с")

# Запрос один и тот же; тексты показываем для первого измеренного запуска.
# Полные результаты всех запусков сохранены в response['runs'].
result = response['runs'][0]['result']
print(f"\nДокументов возвращено: {result['returned']}")
for rank, doc in enumerate(result['candidates'], 1):
    print(f"\n{rank}. {doc['doc_id']} | score={doc['score']}")
    print(doc['title'] if doc['title'] is not None else '(title не сохранён в индексе)')
    print(doc['text'] if doc['text'] is not None else '(text не сохранён в индексе)')
